* # 1. Importación de Librerías y Dependencias *
Este bloque de código se encarga de importar todas las librerías y módulos necesarios de TensorFlow/Keras y utilidades de Python para
construir, entrenar y gestionar el modelo de aprendizaje profundo.

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2  # ✅ AGREGAR ESTA LÍNEA
import matplotlib.pyplot as plt
import numpy as np
import os
import json

print("✅ Librerías importadas")
print("TensorFlow version:", tf.__version__)

✅ Librerías importadas
TensorFlow version: 2.20.0


# 2. Configuración de Parámetros y Rutas del Dataset
En este bloque se definen los hiperparámetros clave del modelo y se establece la ruta base donde se encuentran las carpetas con las imágenes del dataset.

In [3]:
# Parámetros del modelo OPTIMIZADOS
IMG_HEIGHT = 160   # Balance velocidad/calidad
IMG_WIDTH = 160    
BATCH_SIZE = 16    
EPOCHS_PHASE1 = 3   # Fase 1 reducida
EPOCHS_PHASE2 = 12  # Fase 2 reducida
NUM_CLASSES = 10

# Rutas - USANDO TU SISTEMA PROBADO
BASE_DIR = 'talento_tech' 

# Verificar que la carpeta existe (TU CÓDIGO FUNCIONA)
if os.path.exists(BASE_DIR):
    BASE_DIR = os.path.abspath(BASE_DIR)
    print(f"✅ Carpeta encontrada: {BASE_DIR}")
    print("Contenido:", os.listdir(BASE_DIR))
else:
    cwd = os.getcwd()
    if os.path.basename(cwd) == BASE_DIR:
        BASE_DIR = cwd
        print(f"✅ Ejecutando dentro de la carpeta: {BASE_DIR}")
        print("Contenido:", os.listdir(BASE_DIR))
    else:
        found = None
        path = cwd
        while True:
            candidate = os.path.join(path, BASE_DIR)
            if os.path.exists(candidate):
                found = os.path.abspath(candidate)
                break
            parent = os.path.dirname(path)
            if parent == path:
                break
            path = parent
        if found:
            BASE_DIR = found
            print(f"✅ Carpeta encontrada: {BASE_DIR}")
            print("Contenido:", os.listdir(BASE_DIR))
        else:
            print(f"❌ Carpeta NO encontrada: {BASE_DIR}")
            print("Directorio actual:", cwd)
            print("Sugerencia: ajusta BASE_DIR a la ruta correcta")

✅ Ejecutando dentro de la carpeta: c:\xampp\htdocs\talento_tech
Contenido: ['bolsas_plastico', 'botellas_plastico', 'envases_plastico', 'mejor_modelo_mobilenet.h5', 'modelo.ipynb', 'modelo2.ipynb', 'organic', 'papel', 'recyclable', 'residuos_medicos', 'tetra_pack', 'vasos_papel', 'vidrio']


# 3. Aumentación de Datos y Generadores
Este código configura la Aumentación de Datos (Data Augmentation) y crea los Generadores de Datos (ImageDataGenerator) que cargarán, preprocesarán y aumentarán las imágenes en tiempo real.

In [4]:
# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Generadores
train_generator = train_datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    BASE_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("✅ Generadores creados:")
print(f"   - Entrenamiento: {train_generator.samples} imágenes")
print(f"   - Validación: {validation_generator.samples} imágenes")
print(f"   - Clases: {train_generator.class_indices}")

Found 24739 images belonging to 10 classes.
Found 6182 images belonging to 10 classes.
✅ Generadores creados:
   - Entrenamiento: 24739 imágenes
   - Validación: 6182 imágenes
   - Clases: {'bolsas_plastico': 0, 'botellas_plastico': 1, 'envases_plastico': 2, 'organic': 3, 'papel': 4, 'recyclable': 5, 'residuos_medicos': 6, 'tetra_pack': 7, 'vasos_papel': 8, 'vidrio': 9}


# 4. Creación y Compilación del Modelo MobileNetV2
Este bloque define la arquitectura del modelo de Transfer Learning. Se carga la red MobileNetV2 pre-entrenada y se le añade una "cabeza" de clasificación personalizada.

In [7]:
def create_mobilenet_model():
    base_model = MobileNetV2(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    base_model.trainable = False
    
    model = Sequential([
        base_model,
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(len(train_generator.class_indices), activation='softmax')
    ])
    return model

# Crear y compilar modelo
model = create_mobilenet_model()
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Modelo MobileNetV2 creado")
model.summary()

✅ Modelo MobileNetV2 creado


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160            │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# 5. Definición de Callbacks de Entrenamiento
Los Callbacks son funciones que se ejecutan durante el entrenamiento para automatizar ciertas acciones, como detener el entrenamiento temprano o ajustar la tasa de aprendizaje.

In [8]:
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        mode='max'
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'mejor_modelo_mobilenet.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max'
    )
]

print("✅ Callbacks configurados")

✅ Callbacks configurados


# 6. Fase 1: Entrenamiento Base (Modelo Congelado)
Se inicia la primera fase de entrenamiento. Como el modelo MobileNetV2 está congelado, solo se entrenan las capas personalizadas añadidas en la "cabeza" de clasificación.

In [9]:
print("🎯 FASE 1: Entrenamiento Base (Modelo Congelado)")
print("=" * 50)

history_phase1 = model.fit(
    train_generator,
    epochs=EPOCHS_PHASE1,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("✅ Fase 1 completada")

🎯 FASE 1: Entrenamiento Base (Modelo Congelado)


c:\Users\57323\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3
  15/1547 ━━━━━━━━━━━━━━━━━━━━ 1:05:29 3s/step - accuracy: 0.3249 - loss: 2.1273

KeyboardInterrupt: 

In [10]:
# CELDA EXTRA - CARGAR MODELO ENTRENADO
import tensorflow as tf

# Cargar el modelo que YA completó FASE 1
model = tf.keras.models.load_model('mejor_modelo_mobilenet.h5')
print("✅ Modelo de FASE 1 cargado (77-80% accuracy)")

# Verificar que carga correctamente
test_loss, test_accuracy = model.evaluate(validation_generator, verbose=0)
print(f"✅ Precisión del modelo cargado: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

✅ Modelo de FASE 1 cargado (77-80% accuracy)


c:\Users\57323\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Precisión del modelo cargado: 0.8198 (81.98%)


# 8. Fase 2: Ejecución del Fine-Tuning
Se ejecuta la segunda fase de entrenamiento, donde toda la red (MobileNetV2 y la cabeza de clasificación) se ajusta para el dataset específico de residuos.

In [14]:
print("🎯 INICIANDO FASE 2 - FINE TUNING")
print("=" * 50)
print("Accuracy inicial: ~81%")
print("Accuracy objetivo: 85-88%")
print("Tiempo estimado: 1.5-2 horas")

# Descongelar capas para fine-tuning
base_model = model.layers[0]
base_model.trainable = True

# Congelar primeras capas, descongelar últimas 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Re-compilar con learning rate más bajo para fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("🔓 Capas descongeladas para fine-tuning")

# Callbacks MEJORADOS para guardar progreso
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=6,  # Un poco más de paciencia
        restore_best_weights=True,
        mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'modelo_final_residuos.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# INICIAR FASE 2 CON 8 ÉPOCAS
history_phase2 = model.fit(
    train_generator,
    epochs=8,  # ← 8 ÉPOCAS COMO DECIDISTE
    validation_data=validation_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

print("✅ FASE 2 COMPLETADA - Modelo listo para producción")

🎯 INICIANDO FASE 2 - FINE TUNING
Accuracy inicial: ~81%
Accuracy objetivo: 85-88%
Tiempo estimado: 1.5-2 horas
🔓 Capas descongeladas para fine-tuning
Epoch 1/8
  30/1547 ━━━━━━━━━━━━━━━━━━━━ 23:15 920ms/step - accuracy: 0.8918 - loss: 0.3218

c:\Users\57323\AppData\Local\Programs\Python\Python311\Lib\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8519 - loss: 0.4319
Epoch 1: val_accuracy improved from None to 0.76286, saving model to modelo_final_residuos.h5


1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1994s 1s/step - accuracy: 0.8476 - loss: 0.4422 - val_accuracy: 0.7629 - val_loss: 0.6894 - learning_rate: 1.0000e-04
Epoch 2/8
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 995ms/step - accuracy: 0.8486 - loss: 0.4299
Epoch 2: val_accuracy improved from 0.76286 to 0.77564, saving model to modelo_final_residuos.h5


1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1839s 1s/step - accuracy: 0.8523 - loss: 0.4300 - val_accuracy: 0.7756 - val_loss: 0.6420 - learning_rate: 1.0000e-04
Epoch 3/8
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 812ms/step - accuracy: 0.8618 - loss: 0.3892
Epoch 3: val_accuracy improved from 0.77564 to 0.79262, saving model to modelo_final_residuos.h5


1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1498s 969ms/step - accuracy: 0.8587 - loss: 0.4051 - val_accuracy: 0.7926 - val_loss: 0.6332 - learning_rate: 1.0000e-04
Epoch 4/8
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 790ms/step - accuracy: 0.8629 - loss: 0.3801
Epoch 4: val_accuracy improved from 0.79262 to 0.81948, saving model to modelo_final_residuos.h5


1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1450s 937ms/step - accuracy: 0.8658 - loss: 0.3798 - val_accuracy: 0.8195 - val_loss: 0.5634 - learning_rate: 1.0000e-04
Epoch 5/8
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 782ms/step - accuracy: 0.8734 - loss: 0.3681
Epoch 5: val_accuracy did not improve from 0.81948
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1440s 931ms/step - accuracy: 0.8700 - loss: 0.3786 - val_accuracy: 0.8138 - val_loss: 0.5909 - learning_rate: 1.0000e-04
Epoch 6/8
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8788 - loss: 0.3470
Epoch 6: val_accuracy did not improve from 0.81948
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 2155s 1s/step - accuracy: 0.8749 - loss: 0.3575 - val_accuracy: 0.8149 - val_loss: 0.6149 - learning_rate: 1.0000e-04
Epoch 7/8
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 0s 842ms/step - accuracy: 0.8784 - loss: 0.3483
Epoch 7: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 7: val_accuracy did not improve from 0.81948
1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1533s 986ms/step - acc

1547/1547 ━━━━━━━━━━━━━━━━━━━━ 1413s 913ms/step - accuracy: 0.8893 - loss: 0.3138 - val_accuracy: 0.8251 - val_loss: 0.5280 - learning_rate: 5.0000e-05
✅ FASE 2 COMPLETADA - Modelo listo para producción


# 10. Guardado Final, Configuración y Evaluación
El último bloque guarda la mejor versión del modelo entrenado y la configuración de las clases para su uso futuro, y realiza una evaluación final de la precisión.

In [15]:
# === CELDA FINAL: GUARDAR CONFIGURACIÓN COMPLETA ===

import json
import tensorflow as tf

# 1. Guardar configuración de clases
class_config = {
    'class_indices': train_generator.class_indices,
    'img_height': IMG_HEIGHT,
    'img_width': IMG_WIDTH,
    'accuracy_final': 0.8251,
    'classes': list(train_generator.class_indices.keys())
}

with open('configuracion_clases.json', 'w') as f:
    json.dump(class_config, f, indent=2)

print("✅ configuracion_clases.json guardado")

# 2. Cargar el MEJOR modelo final
model_final = tf.keras.models.load_model('modelo_final_residuos.h5')

# 3. Evaluación final de precisión
print("📊 EVALUACIÓN FINAL:")
test_loss, test_accuracy = model_final.evaluate(validation_generator, verbose=0)
print(f"🎯 PRECISIÓN FINAL: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# 4. Información del modelo
print("\n📁 ARCHIVOS GENERADOS:")
import os
for file in os.listdir('.'):
    if '.h5' in file or '.json' in file:
        print(f"   - {file}")

print("\n🎉 ¡MODELO LISTO PARA PRODUCCIÓN!")
print("🔧 Puedes integrarlo en tu sitio web")
print("📱 82.51% de precisión - Clasifica 8 de cada 10 residuos correctamente")

✅ configuracion_clases.json guardado


📊 EVALUACIÓN FINAL:
🎯 PRECISIÓN FINAL: 0.8253 (82.53%)

📁 ARCHIVOS GENERADOS:
   - configuracion_clases.json
   - mejor_modelo_mobilenet.h5
   - modelo_final_residuos.h5

🎉 ¡MODELO LISTO PARA PRODUCCIÓN!
🔧 Puedes integrarlo en tu sitio web
📱 82.51% de precisión - Clasifica 8 de cada 10 residuos correctamente
